
# Worker activity recognition

This Notebook provides a comprehensive workflow for worker activity recognition and classification in **Pick** and **NoPick** class. It can train the model in both GPU and CPU environment, and save the trained model. Additionally, it supports evaluating a pre-trained model on a test dataset.
- **Training a Model**: Set `train_val_test=True` to enable training and validation.
- **Evaluating a Pre-trained Model**: Set `train_val_test=False` to skip training and directly evaluate a pre-trained model.


In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import tensorflow as tf
tf.get_logger().setLevel('ERROR')
from tensorflow import keras
import pandas as pd
import seaborn as sns
from pylab import rcParams
import matplotlib.pyplot as plt
from matplotlib import rc
from pandas.plotting import register_matplotlib_converters
from tensorflow.keras.callbacks import ModelCheckpoint
from datetime import datetime
%matplotlib inline
%config InlineBackend.figure_format='retina'

from yieldnn2025 import YieldNN
from utils.misc import VelocityCalculator, CreateDataset

from sklearn.preprocessing import OneHotEncoder,RobustScaler
from sklearn.metrics import confusion_matrix,precision_recall_curve, auc


import pytz


register_matplotlib_converters()
sns.set(style='whitegrid', palette='muted', font_scale=1.5)

rcParams['figure.figsize'] = 22, 10

RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

In [ ]:
device_name=tf.test.gpu_device_name()
if device_name:
    print(f"GPU Available {device_name}")
else:
    print(f"No GPU !! {device_name}")

In [ ]:
# Specify if using a trained model or training a new one
train_val_test=False  
if not train_val_test:
    #define the trained model path
    checkpoint_path="./weights/mass_imu_0210-1657"


### Reading and Combining Annotated Data

The annotated data from multiple CSV files is read and combined into a single DataFrame.

In [ ]:

column_names = ['date_cartID','GPS_TOW','easting','northing', 'ax', 'ay', 'az', 'raw_mass', 'activity']
data_root="./datasets"
# Define the file paths and corresponding column names
file_paths = [
    "4-10-24_train-ready_all_carts.csv",
    "4-17-24_train-ready_all_carts.csv",
    "4-24-24_train-ready_all_carts.csv",
    "4-27-24_train-ready_all_carts.csv",
    "5-1-24_train-ready_all_carts.csv",
    "5-4-24_train-ready_all_carts.csv",
    "5-7-24_train-ready_all_carts.csv",
    "5-11-24_train-ready_all_carts.csv",
    "5-15-24_train-ready_all_carts.csv",
    "5-18-24_train-ready_all_carts.csv",
    "5-22-24_train-ready_all_carts.csv",
    "5-25-24_train-ready_all_carts.csv"
]
test_data="5-15-24" #"4-24-24", "5-4-24", "5-11-24", "5-15-24", "5-25-24"

dfs = []
for file_name in file_paths:
    file_path = os.path.join(data_root, "SantaMaria_CNNLSTM","cart_data", file_name)
    if os.path.exists(file_path):
        df_temp = pd.read_csv(file_path, usecols=column_names, on_bad_lines='skip').dropna()
        dfs.append(df_temp)
        print(f"Loaded file: {file_path} with shape {df_temp.shape}")
    else:
        print(f"File not found: {file_path}")

df = pd.concat(dfs).dropna()

In [ ]:
# calculate velocity
velocity_calculator = VelocityCalculator(df[['GPS_TOW', 'easting', 'northing']])
df["vel"],_, _ = velocity_calculator.calculate_velocity()
df = df.reset_index(drop=True).dropna()


In [ ]:
df.head()

In [ ]:
print(df.describe())

In [ ]:
plt.figure(figsize=(10, 8))
sns.countplot(
    x='activity',
    data=df,
    order=df.activity.value_counts().index,
    palette='Set2'
)
plt.title('Activity Distribution')
plt.xlabel('Activity')
plt.ylabel('Count')
plt.show()

In [ ]:
# Split the data into training and test sets based on the date_cartID
df_train=df[~df['date_cartID'].str.startswith(test_data)]
df_test=df[df['date_cartID'].str.startswith(test_data)]
df_train = df_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)
print(f"Dataset shape: {df.shape},Train shape: {df_train.shape}, Test shape: {df_test.shape}")

In [ ]:
train_mode= "mass_imu" #mass_imu_vel, mass_imu, mass, imu,vel

if train_mode=="mass_imu_vel":
    train_columns = ['vel','ax', 'ay', 'az','raw_mass']
if train_mode=="mass_imu":
    train_columns = ['ax', 'ay', 'az','raw_mass']
if train_mode=="mass":
    train_columns = ['raw_mass']
if train_mode=="imu":
    train_columns = ['ax', 'ay', 'az']
if train_mode=="vel":
    train_columns = ['vel']


In [ ]:
# scaling the data
scaler = RobustScaler()
scaler = scaler.fit(df_train[train_columns])

df_train.loc[:, train_columns] = scaler.transform(df_train[train_columns].to_numpy())
df_test.loc[:, train_columns] = scaler.transform(df_test[train_columns].to_numpy())

In [ ]:
TIME_STEPS = 9600  # Number of time steps in each segment
STEP_SIZE = 5000   # Step size for sliding window

X_train, y_train = CreateDataset.create_dataset(
    df_train[train_columns],
    df_train.activity,
    TIME_STEPS,
    STEP_SIZE
)

X_test, y_test = CreateDataset.create_dataset(
    df_test[train_columns],
    df_test.activity,
    TIME_STEPS,
    STEP_SIZE
)

In [ ]:
print(X_train.shape, y_train.shape)
print(X_test.shape,y_test.shape)

In [ ]:

enc = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

enc = enc.fit(y_train.reshape(-1,1))

y_train_encoded = enc.transform(y_train.reshape(-1,1))
y_train=y_train_encoded.reshape(-1,96,2)
y_test_encoded = enc.transform(y_test.reshape(-1,1))
y_test=y_test_encoded.reshape(-1,96,2)


In [ ]:
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

In [ ]:

if train_val_test:
    # Can actually define the input shape based on the train data. However, keras requires better serialization. 
    # For now, if using different input shapes it is better to define the input shape explicitly in yieldnn2025.py
    # Example: mass_imu input shape is (9600, 4)
    model=YieldNN()
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['acc'])

    #Defining checkpoint callback
    now = datetime.now(pytz.utc).astimezone(pytz.timezone('America/Los_Angeles'))
    checkpoint_path = os.path.join("./weights",train_mode+"_"+now.strftime("%m%d-%H%M"))
    print(f"Checkpoint file: {checkpoint_path}.keras")
    checkpoint_callback = ModelCheckpoint(checkpoint_path+".keras",
                                          monitor='val_loss',
                                          save_best_only=True,
                                          mode='min',
                                          verbose=1)
    
    # Fit the model
    # Model was trained in Google Colab with GPU, so we will use the same GPU settings. CPU training is also supported.
    if device_name:
        with tf.device(device_name):
        
            history = model.fit(
                X_train, y_train,
                epochs=50,
                batch_size=270,
                validation_split=0.2,
                shuffle=True,
                callbacks=[checkpoint_callback]
            )
    else:
        history = model.fit(
            X_train, y_train,
            epochs=15,
            batch_size=60,
            validation_split=0.2,
            shuffle=True,
            callbacks=[checkpoint_callback]
        )

    

In [ ]:
if train_val_test:
    plt.plot(history.history['loss'], label='train')
    plt.plot(history.history['val_loss'], label='test')
    plt.legend()

# Model Training Completed

In [ ]:
# Load the best model from the checkpoint

model = tf.keras.models.load_model(checkpoint_path + ".keras") 


In [ ]:
if device_name:
    with tf.device(device_name):
        model.evaluate(X_test, y_test)
        y_pred = model.predict(X_test)
else:

    model.evaluate(X_test, y_test)
    y_pred = model.predict(X_test)

In [ ]:
# Defining the confusion matrix plot function
def plot_cm(y_true, y_pred, class_names):
  cm = confusion_matrix(y_true, y_pred)
  fig, ax = plt.subplots(figsize=(16, 12))
  ax = sns.heatmap(
      cm,
      annot=True,
      fmt="d",
      cmap=sns.diverging_palette(220, 20, n=7),
      ax=ax
  )

  plt.ylabel('Actual')
  plt.xlabel('Predicted')
  ax.set_xticklabels(class_names)
  ax.set_yticklabels(class_names)
  b, t = plt.ylim()
  b += 0.5 
  t -= 0.5 
  plt.ylim(b, t) 
  plt.show() 
  return cm


In [ ]:
y_test_flat = y_test.reshape(-1, 2)
y_pred_flat = y_pred.reshape(-1, 2)

In [ ]:
# plotting confusion matrix
conf_matrix=plot_cm(
  enc.inverse_transform(y_test_flat),
  enc.inverse_transform(y_pred_flat),
  enc.categories_[0]
)

class_categories=enc.categories_[0]

In [ ]:
# precision recall F1 score metrics
print(f"Confusion matrix with categories {class_categories}:")
print(conf_matrix)
picking_precision=conf_matrix[1,1]/(conf_matrix[1,1]+conf_matrix[0,1]) # (TP/TP+FP) Precision, considers events classified as picking but were actually not picking
picking_recall=conf_matrix[1,1]/(conf_matrix[1,1]+conf_matrix[1,0]) #TP/TP+FN Recall, considers events classified as not picking but were actually picking
F1=2*(picking_precision*picking_recall)/(picking_precision+picking_recall)
print(f"Training Parameters: {train_columns}")
print(f"Picking precision {picking_precision} recall {picking_recall} F1 {F1}")

In [ ]:
# Plot of precision recall curve
n_classes = y_test_flat.shape[1]

precision = dict()
recall = dict()
average_precision = dict()

for i in range(n_classes):
    precision[i], recall[i], _ = precision_recall_curve(y_test_flat[:, i], y_pred_flat[:, i])
    average_precision[i] = auc(recall[i], precision[i]) 

labels=["NoPick","Pick"]
plt.figure(figsize=(10, 8))
for i in range(n_classes):
    plt.plot(recall[i], precision[i], label=f'{labels[i]} (AP = {average_precision[i]:0.2f})')

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.show()


End of CNN-LSTM model training for worker activity recognition in **Pick** and **NoPick** class is completed. The trained model is saved for future use.